# RealThor v0.1 — Solution

The system prompt is loaded from `prompts/system_prompt.md` rather than written inline.

In [1]:
import os
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
provider = os.getenv("LLM_PROVIDER", "openai")
if provider == "vocareum":
    client = OpenAI(base_url="https://openai.vocareum.com/v1", api_key=os.getenv("VOCAREUM_API_KEY"))
else:
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [4]:
demand = pd.read_csv("data/demand_signals.csv")
supply = pd.read_csv("data/supply_snapshot.csv")

In [5]:
# Load the prompt from the versioned file
with open("prompts/system_prompt.md") as f:
    SYSTEM_PROMPT = f.read()

print("Prompt loaded from prompts/system_prompt.md")

Prompt loaded from prompts/system_prompt.md


In [6]:
def build_context(region: str, property_type: str, bedrooms: int) -> str:
    d = demand[
        (demand["region"] == region)
        & (demand["property_type"] == property_type)
        & (demand["bedrooms"] == bedrooms)
    ]
    s = supply[
        (supply["region"] == region)
        & (supply["property_type"] == property_type)
        & (supply["bedrooms"] == bedrooms)
    ]

    if d.empty or s.empty:
        return "No data available for the requested segment."

    return (
        f"Region: {region}\n"
        f"Property: {bedrooms}-bedroom {property_type}\n\n"
        f"Demand signals:\n{d.to_string(index=False)}\n\n"
        f"Supply snapshot:\n{s.to_string(index=False)}"
    )

In [7]:
def ask_realthor(region: str, property_type: str, bedrooms: int) -> str:
    context = build_context(region, property_type, bedrooms)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": context},
        ],
    )
    return response.choices[0].message.content

In [8]:
recommendation = ask_realthor(region="west", property_type="apartment", bedrooms=2)

In [9]:
print(recommendation)

**Recommendation:**

- **Priority:** High
- **Reasoning Based on Demand:** The demand signals indicate a high level of interest for 2-bedroom apartments in the west region, with a substantial search volume of 1240. The demand score of 0.91 suggests that this property type is in high demand. The lead count of 86 and the number of recent sales at 34 also reflect robust interest and activity in the market.
  
- **Reasoning Based on Supply:** However, the supply snapshot reveals a low number of active listings (only 18) and a very low supply score of 0.22. With an average of just 21 days on the market, this implies that properties are selling quickly, indicating that current supply is insufficient to meet the demand.

- **Recommended Action:** The company should prioritize bringing more 2-bedroom apartments to the platform in the west region to capitalize on strong demand and low supply.

- **Confidence:** High
